# Defense Evaluation — Final Pipeline

**Fixes applied vs previous version:**
- Reports **mAP50** (not mAP50-95) to match the adversarial robustness spreadsheet
- Defense is applied to **both clean AND patched** images before evaluation
- All 4 defenses + none use the same code path (no shortcuts for none)
- Deltas computed per-model relative to that model's own undefended baseline

**Protocol:**
```
IMGSZ=640  CONF_MIN=0.001  IOU=0.6  THRESHOLDS=0.3,0.5,0.7  GT_IOU_MATCH=0.5
Scenarios: CV=billboard01-03  FINAL=billboard04-09
```

## Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics pillow opencv-python numpy pandas matplotlib

import ultralytics, torch, sys
print("Python     :", sys.version)
print("Ultralytics:", ultralytics.__version__)
print("Torch      :", torch.__version__)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## Paths & Config
**Edit only this cell.**

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import torch

random.seed(0); np.random.seed(0); torch.manual_seed(0)

# ── Data roots ─────────────────────────────────────────────────────────────
CLEAN_ROOT     = "/content/drive/MyDrive/Patched Datasets/Clean"
PATCHED_PARENT = "/content/drive/MyDrive/Patched Datasets"   # contains patched_dataset1..9
OUT_ROOT       = "/content/drive/MyDrive/unified_eval_ab"

MODEL_PATHS = {
    "yolov8n": "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt",
    "yolov8s": "/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt",
    "yolov5n": "/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt",
}
ACTIVE_MODELS = ["yolov8n", "yolov8s", "yolov5n"]

# ── Evaluation protocol (fixed) ─────────────────────────────────────────────
IMGSZ        = 640      # must match your spreadsheet
CONF_MIN     = 0.001
IOU          = 0.6
THRESHOLDS   = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
DET_K        = "1,5,10,20,50"
DEVICE       = "0"

# ── Scenario splits ─────────────────────────────────────────────────────────
CV_SCENARIOS    = "1,2,3"        # hyperparameter tuning
FINAL_SCENARIOS = "4,5,6,7,8,9" # held-out final evaluation

# ── Where defended datasets are stored ──────────────────────────────────────
DEFENDED_ROOT = Path("/content/drive/MyDrive/Patched Datasets/Defended_Final_v2")
DEFENDED_ROOT.mkdir(parents=True, exist_ok=True)

# ── Hardcoded best configs from CV sweep (billboard01-03) ───────────────────
# Selection rule: clean drop ≤ 30%, then min Det/img → min FP/img → max patched mAP
FINAL_CONFIGS = [
    ("none",     {}),
    ("gauss",    {"sigma": 4.0}),
    ("median",   {"ksize": 13}),
    ("jpeg",     {"quality": 95}),
    ("bitdepth", {"bits": 7}),
]

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print(f"CLEAN_ROOT    : {CLEAN_ROOT}")
print(f"PATCHED_PARENT: {PATCHED_PARENT}")
print(f"OUT_ROOT      : {OUT_ROOT}")
print(f"Active models : {ACTIVE_MODELS}")
print(f"Final configs : {[c[0] for c in FINAL_CONFIGS]}")

## Write `evaluation_ab.py`

Key fixes:
- Uses `model.val()` Python API (no log parsing)
- Reports **mAP50** as primary metric (+ keeps mAP50-95)
- `imgsz` passed explicitly to `model.predict` (no hardcoded 640)

In [ ]:
%%writefile /content/evaluation_ab.py
import argparse, json, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ─── utils ────────────────────────────────────────────────────────────────
def mean_std(values):
    vals = [v for v in values if isinstance(v, (int, float)) and not np.isnan(v)]
    if not vals: return {"mean": None, "std": None, "n": 0}
    a = np.asarray(vals, dtype=np.float32)
    return {"mean": float(a.mean()),
            "std":  float(a.std(ddof=1)) if len(a) > 1 else 0.0,
            "n":    int(len(a))}

def xywhn_to_xyxy(x, y, w, h):
    return x - w/2, y - h/2, x + w/2, y + h/2

def iou_xyxy(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2]-a[0])*(a[3]-a[1]); ub = (b[2]-b[0])*(b[3]-b[1])
    denom = ua + ub - inter
    return inter / denom if denom > 0 else 0.0

def read_gt(txt_path):
    if not Path(txt_path).exists(): return []
    out = []
    for ln in Path(txt_path).read_text(errors="ignore").strip().splitlines():
        p = ln.strip().split()
        if len(p) >= 5:
            try: out.append((int(float(p[0])), *map(float, p[1:5])))
            except: pass
    return out

def model_names_as_list(model):
    names = getattr(model, "names", None)
    if names is None: raise ValueError("Model has no .names")
    if isinstance(names, dict):
        mk = max(int(k) for k in names)
        out = [None] * (mk + 1)
        for k, v in names.items(): out[int(k)] = str(v)
        return [v or f"class_{i}" for i, v in enumerate(out)]
    return [str(x) for x in names]

def write_yaml(out_yaml, root, img_subpath, names):
    nc   = len(names)
    text = (
        f"path: {root}\n"
        f"train: {img_subpath}\n"
        f"val:   {img_subpath}\n"
        f"test:  {img_subpath}\n"
        f"nc: {nc}\nnames: {names}\n"
    )
    Path(out_yaml).parent.mkdir(parents=True, exist_ok=True)
    Path(out_yaml).write_text(text)

# ─── YOLO val via Python API — returns BOTH mAP50 and mAP50-95 ────────────
def run_yolo_val(model, yaml_path, imgsz, conf, iou, device):
    results = model.val(
        data=str(yaml_path), imgsz=imgsz, conf=conf, iou=iou,
        device=device, split="test", save_json=False, plots=False, verbose=False,
    )
    rd = results.results_dict if hasattr(results, "results_dict") else {}

    def _get(*keys):
        for k in keys:
            if k in rd and rd[k] is not None:
                return float(rd[k])
        # fallback to box attribute
        box = getattr(results, "box", None)
        if box is not None:
            for attr in ("map50", "map", "mp", "mr"):
                v = getattr(box, attr, None)
                if v is not None: return float(v)
        return None

    return {
        "mAP50":     _get("metrics/mAP50(B)",    "mAP50"),     # PRIMARY
        "mAP50-95":  _get("metrics/mAP50-95(B)", "mAP50-95"),  # kept for reference
        "precision": _get("metrics/precision(B)", "precision"),
        "recall":    _get("metrics/recall(B)",    "recall"),
    }

# ─── Density + FP metrics — imgsz passed explicitly ───────────────────────
def evaluate_density(model, img_dir, lbl_dir, conf_min, iou_thresh,
                     thresholds, gt_iou_match, det_k, imgsz=640):
    img_dir  = Path(img_dir); lbl_dir = Path(lbl_dir)
    imgs     = sorted(p for p in img_dir.rglob("*")
                      if p.is_file() and p.suffix.lower() in IMG_EXTS)
    if not imgs:
        print(f"  WARNING: no images in {img_dir}")

    det_counts, fp_conf_min = [], []
    fp_thr  = {t: [] for t in thresholds}
    det_thr = {t: [] for t in thresholds}
    topk    = {k: [] for k in det_k}

    for img_path in imgs:
        gts = read_gt(lbl_dir / (img_path.stem + ".txt"))
        res = list(model.predict(
            source=str(img_path), imgsz=imgsz,
            conf=conf_min, iou=iou_thresh,
            verbose=False, max_det=10000))
        if not res or res[0].boxes is None or len(res[0].boxes) == 0:
            det_counts.append(0); fp_conf_min.append(0)
            for t in thresholds: fp_thr[t].append(0); det_thr[t].append(0)
            for k in det_k: topk[k].append(0)
            continue

        confs = res[0].boxes.conf.cpu().numpy()
        xyxyn = res[0].boxes.xyxyn.cpu().numpy()
        det_counts.append(int(len(confs)))

        def count_fps(mask):
            matched = set(); fps = 0
            for pb in xyxyn[mask]:
                hit = False
                for gi, gt in enumerate(gts):
                    if gi in matched: continue
                    if iou_xyxy(pb, xywhn_to_xyxy(*gt[1:5])) >= gt_iou_match:
                        matched.add(gi); hit = True; break
                if not hit: fps += 1
            return fps

        fp_conf_min.append(count_fps(np.ones(len(confs), dtype=bool)))
        for t in thresholds:
            mask = confs >= t
            det_thr[t].append(int(mask.sum()))
            fp_thr[t].append(count_fps(mask))
        for k in det_k:
            topk[k].append(min(int(len(confs)), k))

    n = max(len(imgs), 1)
    A = {
        "images_eval":               n,
        "detections_per_image_mean": float(np.mean(det_counts)) if det_counts else 0.0,
        "total_detections_conf_ge":  {str(t): int(sum(det_thr[t])) for t in thresholds},
        "topk_total":                {str(k): int(sum(topk[k]))    for k in det_k},
    }
    B = {
        "fp_per_image_mean_conf_min": float(np.mean(fp_conf_min)) if fp_conf_min else 0.0,
        "fp_per_image_conf_ge":       {str(t): float(np.mean(fp_thr[t])) for t in thresholds},
    }
    return A, B

# ─── Auto-detect image/label dirs ─────────────────────────────────────────
def resolve_dirs(root, scenario):
    candidates = [
        (root / "images" / scenario,           root / "labels" / scenario,           f"images/{scenario}"),
        (root / "images" / "test" / scenario,  root / "labels" / "test" / scenario,  f"images/test/{scenario}"),
        (root / "images" / "test",             root / "labels" / "test",             "images/test"),
        (root / "images",                      root / "labels",                      "images"),
    ]
    for img_dir, lbl_dir, subpath in candidates:
        if img_dir.exists() and any(True for p in img_dir.rglob("*")
                                    if p.is_file() and p.suffix.lower() in IMG_EXTS):
            return img_dir, lbl_dir, subpath
    return None, None, None

# ─── Main ─────────────────────────────────────────────────────────────────
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model",           required=True)
    ap.add_argument("--clean_root",      required=True)
    ap.add_argument("--patched_root",    default=None)
    ap.add_argument("--patched_parent",  default=None)
    ap.add_argument("--split",           default="test")
    ap.add_argument("--scenarios",       default="1,2,3")
    ap.add_argument("--out_root",        required=True)
    ap.add_argument("--run_name",        required=True)
    ap.add_argument("--imgsz",           type=int,   default=640)
    ap.add_argument("--conf_min",        type=float, default=0.001)
    ap.add_argument("--iou",             type=float, default=0.6)
    ap.add_argument("--thresholds",      default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match",    type=float, default=0.5)
    ap.add_argument("--det_k",           default="1,5,10,20,50")
    ap.add_argument("--device",          default="0")
    args = ap.parse_args()

    thresholds = [float(x) for x in args.thresholds.split(",")]
    det_k      = [int(x)   for x in args.det_k.split(",")]
    scenarios  = []
    for tok in args.scenarios.split(","):
        tok = tok.strip()
        scenarios.append(tok if tok.startswith("billboard")
                         else f"billboard{int(tok):02d}")

    use_patched_parent = (args.patched_root is None and args.patched_parent is not None)

    print(f"Loading model: {args.model}")
    model   = YOLO(args.model)
    names   = model_names_as_list(model)
    run_dir = Path(args.out_root) / args.run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    per_scenario = []
    collector    = {"clean": {}, "patched": {}}

    for scen in scenarios:
        scen_idx = int(scen.replace("billboard", ""))
        print(f"\n{'='*60}\n  Scenario: {scen}\n{'='*60}")
        result = {"scenario": scen}

        clean_root = Path(args.clean_root)
        if use_patched_parent:
            patched_root = Path(args.patched_parent) / f"patched_dataset{scen_idx}"
        else:
            patched_root = Path(args.patched_root)

        for cond, root in (("clean", clean_root), ("patched", patched_root)):
            img_dir, lbl_dir, img_subpath = resolve_dirs(root, scen)
            print(f"  [{cond}] {img_dir}  exists:{img_dir.exists() if img_dir else False}")
            if img_dir is None or not img_dir.exists():
                print(f"  SKIP — no images found under {root}")
                continue

            yaml_path = run_dir / f"{scen}_{cond}.yaml"
            write_yaml(yaml_path, root, img_subpath, names)
            yolo_metrics = run_yolo_val(model, yaml_path, args.imgsz,
                                        args.conf_min, args.iou, args.device)
            print(f"    yolo_val: mAP50={yolo_metrics.get('mAP50'):.4f}  "
                  f"mAP50-95={yolo_metrics.get('mAP50-95'):.4f}")

            A, B = evaluate_density(
                model, img_dir, lbl_dir,
                args.conf_min, args.iou,
                thresholds, args.gt_iou_match, det_k,
                imgsz=args.imgsz)

            result[cond] = {"root": str(root), "yolo_val": yolo_metrics, "A": A, "B": B}

            for key in ("mAP50", "mAP50-95", "precision", "recall"):
                collector[cond].setdefault(key, []).append(yolo_metrics.get(key))
            collector[cond].setdefault("det_mean", []).append(A["detections_per_image_mean"])
            collector[cond].setdefault("fp_mean",  []).append(B["fp_per_image_mean_conf_min"])

        if "clean" in result and "patched" in result:
            delta = {}
            for sub in ("yolo_val", "A", "B"):
                delta[sub] = {}
                for k, vc in result["clean"][sub].items():
                    vp = result["patched"][sub].get(k)
                    if isinstance(vc, (int, float)) and isinstance(vp, (int, float)):
                        delta[sub][k] = round(vp - vc, 6)
            result["delta_patched_minus_clean"] = delta

        per_scenario.append(result)

    ms = {}
    for cond in ("clean", "patched"):
        for key, vals in collector[cond].items():
            ms[f"{cond}_{key}"] = mean_std([v for v in vals if v is not None])
    for key in ("mAP50", "mAP50-95"):
        cv = [v for v in collector["clean"].get(key,   []) if v is not None]
        pv = [v for v in collector["patched"].get(key, []) if v is not None]
        if cv and pv and len(cv) == len(pv):
            ms[f"delta_{key}"] = mean_std([p - c for p, c in zip(pv, cv)])

    combined = {
        "model": args.model, "run_name": args.run_name,
        "split": args.split, "scenarios": scenarios,
        "imgsz": args.imgsz, "conf_min": args.conf_min, "iou": args.iou,
        "thresholds": thresholds, "gt_iou_match": args.gt_iou_match,
        "per_scenario": per_scenario, "mean_std": ms,
    }
    out_json = run_dir / "combined_metrics.json"
    out_json.write_text(json.dumps(combined, indent=2))
    print(f"\n✅  Saved: {out_json}")

if __name__ == "__main__":
    main()
print("✅ evaluation_ab.py written")

## Write `defenses_apply.py`

In [ ]:
%%writefile /content/defenses_apply.py
import io, shutil
from pathlib import Path
from typing import Callable, Dict, Optional
import cv2
import numpy as np
from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def defense_none(img, p):     return img

def defense_gauss(img, p):
    return cv2.GaussianBlur(img, (0,0), sigmaX=float(p.get("sigma",1.0)))

def defense_median(img, p):
    k = int(p.get("ksize", 3))
    if k % 2 == 0: k += 1
    return cv2.medianBlur(img, k)

def defense_jpeg(img, p):
    q   = int(p.get("quality", 85))
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format="JPEG", quality=q, optimize=True)
    buf.seek(0)
    return np.array(Image.open(buf).convert("RGB"), dtype=np.uint8)

def defense_bitdepth(img, p):
    bits   = max(1, min(8, int(p.get("bits", 5))))
    levels = 2 ** bits
    x      = img.astype(np.float32) / 255.0
    return (np.round(x*(levels-1))/(levels-1)*255).clip(0,255).astype(np.uint8)

DEFENSES: Dict[str, Callable] = {
    "none":     defense_none,
    "gauss":    defense_gauss,
    "median":   defense_median,
    "jpeg":     defense_jpeg,
    "bitdepth": defense_bitdepth,
}

def apply_defense_to_dir(
    src_img_dir: Path,
    src_lbl_dir: Path,
    dst_img_dir: Path,
    dst_lbl_dir: Path,
    defense_fn:  Callable,
    params:      dict,
    skip_if_exists: bool = True,
) -> int:
    if skip_if_exists and dst_img_dir.exists() and any(dst_img_dir.rglob("*")):
        n = sum(1 for p in dst_img_dir.rglob("*") if p.is_file())
        print(f"    [skip] {dst_img_dir.name} already exists ({n} files)")
        return n

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for src in sorted(src_img_dir.rglob("*")):
        if not (src.is_file() and src.suffix.lower() in IMG_EXTS): continue
        arr = np.array(Image.open(src).convert("RGB"), dtype=np.uint8)
        out = defense_fn(arr, params)
        dst = dst_img_dir / src.relative_to(src_img_dir)
        dst.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(out.astype(np.uint8), mode="RGB").save(dst.with_suffix(".png"))
        count += 1

    if src_lbl_dir.exists():
        if dst_lbl_dir.exists(): shutil.rmtree(dst_lbl_dir)
        shutil.copytree(src_lbl_dir, dst_lbl_dir)

    return count

print("✅ defenses_apply.py written:", list(DEFENSES.keys()))

## Helper Functions

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from defenses_apply import DEFENSES, apply_defense_to_dir

EVAL_SCRIPT = Path("/content/evaluation_ab.py")
assert EVAL_SCRIPT.exists(), "Run Cell 3 first."

def _tag(def_name, params):
    if not params: return def_name
    return def_name + "_" + "_".join(f"{k}{v}" for k, v in sorted(params.items()))

def run_eval(run_name, clean_root, scenarios_str, out_root, model_path,
             patched_root=None, patched_parent=None, device="0"):
    out_root = Path(out_root); out_root.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python3", str(EVAL_SCRIPT),
        "--clean_root",   str(clean_root),
        "--scenarios",    scenarios_str,
        "--out_root",     str(out_root),
        "--run_name",     run_name,
        "--model",        str(model_path),
        "--imgsz",        str(IMGSZ),
        "--conf_min",     str(CONF_MIN),
        "--iou",          str(IOU),
        "--thresholds",   THRESHOLDS,
        "--gt_iou_match", str(GT_IOU_MATCH),
        "--det_k",        DET_K,
        "--device",       str(device),
    ]
    if patched_root:   cmd += ["--patched_root",   str(patched_root)]
    if patched_parent: cmd += ["--patched_parent", str(patched_parent)]
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    for fname in ("combined_metrics.json", "summary.json"):
        p = out_root / run_name / fname
        if p.exists(): return str(p)
    raise FileNotFoundError(f"No JSON in {out_root/run_name}")

def extract_metrics(json_path):
    """Returns (clean_mAP50, patched_mAP50, det_img, fp_img) — all float, 0.0 if missing."""
    data = json.loads(Path(json_path).read_text())
    ms   = data.get("mean_std", {})

    def _g(*keys):
        for k in keys:
            e = ms.get(k)
            if isinstance(e, dict) and e.get("mean") is not None:
                return float(e["mean"])
        return None

    # PRIMARY metric: mAP50 (matches your spreadsheet)
    c = _g("clean_mAP50",   "clean_mAP50_mean")
    p = _g("patched_mAP50", "patched_mAP50_mean")
    d = _g("patched_det_mean")
    f = _g("patched_fp_mean")

    # Fallback: compute from per_scenario if mean_std missing
    if None in (c, p, d, f):
        c_l, p_l, d_l, f_l = [], [], [], []
        for s in data.get("per_scenario", []):
            if "clean"   in s: c_l.append(s["clean"]["yolo_val"].get("mAP50") or 0)
            if "patched" in s:
                p_l.append(s["patched"]["yolo_val"].get("mAP50") or 0)
                d_l.append(s["patched"]["A"].get("detections_per_image_mean") or 0)
                f_l.append(s["patched"]["B"].get("fp_per_image_mean_conf_min") or 0)
        if c_l: c = float(np.mean(c_l))
        if p_l: p = float(np.mean(p_l))
        if d_l: d = float(np.mean(d_l))
        if f_l: f = float(np.mean(f_l))

    return (c or 0.0), (p or 0.0), (d or 0.0), (f or 0.0)

def extract_metrics_full(json_path):
    """Full metrics including per-threshold FP/det."""
    data = json.loads(Path(json_path).read_text())
    ms   = data.get("mean_std", {})

    def _g(*keys):
        for k in keys:
            e = ms.get(k)
            if isinstance(e, dict) and e.get("mean") is not None:
                return float(e["mean"])
        return 0.0

    c = _g("clean_mAP50");  p = _g("patched_mAP50")
    c95 = _g("clean_mAP50-95"); p95 = _g("patched_mAP50-95")

    det  = {t: [] for t in ["0.001","0.3","0.5","0.7"]}
    fp   = {t: [] for t in ["0.001","0.3","0.5","0.7"]}

    for s in data.get("per_scenario", []):
        if "patched" not in s: continue
        A = s["patched"].get("A", {})
        B = s["patched"].get("B", {})
        n = A.get("images_eval", 1)
        det["0.001"].append(A.get("detections_per_image_mean", 0))
        fp ["0.001"].append(B.get("fp_per_image_mean_conf_min", 0))
        for t in ["0.3","0.5","0.7"]:
            det[t].append(A.get("total_detections_conf_ge",{}).get(t,0) / max(n,1))
            fp [t].append(B.get("fp_per_image_conf_ge",{}).get(t,0))

    def _m(lst): return round(float(np.mean(lst)),3) if lst else 0.0

    return dict(
        clean_mAP50=round(c,4),   patched_mAP50=round(p,4),
        clean_mAP5095=round(c95,4), patched_mAP5095=round(p95,4),
        det_0001=_m(det["0.001"]), fp_0001=_m(fp["0.001"]),
        det_03=_m(det["0.3"]),     fp_03=_m(fp["0.3"]),
        det_05=_m(det["0.5"]),     fp_05=_m(fp["0.5"]),
        det_07=_m(det["0.7"]),     fp_07=_m(fp["0.7"]),
    )

print("✅ Helper functions ready")

## Sanity Check

In [ ]:
from pathlib import Path

clean_root     = Path(CLEAN_ROOT)
patched_parent = Path(PATCHED_PARENT)

print("Clean images exists:", (clean_root / "images").exists())
print("Clean labels exists:", (clean_root / "labels").exists())
print()
for i in range(1, 10):
    bb  = f"billboard{i:02d}"
    ci  = clean_root / "images" / bb
    pi  = patched_parent / f"patched_dataset{i}" / "images" / "test"
    nci = len(list(ci.glob("*"))) if ci.exists() else 0
    npi = len(list(pi.glob("*"))) if pi.exists() else 0
    print(f"{bb}  clean:{ci.exists()}({nci})  patched:{pi.exists()}({npi})")

## Final Held-Out Evaluation (billboard04–09)

Applies each defense to both clean and patched images, evaluates with `model.val()`, and reports mAP50, Det/img, and FP/img deltas.

In [ ]:
import time

final_scen_list = [int(x) for x in FINAL_SCENARIOS.split(",")]
clean_root      = Path(CLEAN_ROOT)
all_rows        = []

for model_key in ACTIVE_MODELS:
    model_path     = MODEL_PATHS[model_key]
    model_none_ref = None

    print(f"\n{'#'*70}\n  FINAL — {model_key}\n{'#'*70}")

    for def_name, params in FINAL_CONFIGS:
        tag   = _tag(def_name, params)
        label = f"{model_key}_FINAL_bill4to9_{tag}"
        print(f"\n{'='*60}\n  {label}\n{'='*60}")
        t0 = time.time()

        # ── Apply defense to BOTH clean and patched ─────────────────────────
        # Each model gets its own defended folder to avoid cross-model contamination
        def_dir          = DEFENDED_ROOT / f"{model_key}_{tag}"
        def_clean_root   = def_dir / "clean"
        def_patched_root = def_dir / "patched"

        try:
            for i in final_scen_list:
                bb    = f"billboard{i:02d}"
                p_src = Path(PATCHED_PARENT) / f"patched_dataset{i}"
                p_img = p_src / "images" / "test"
                p_lbl = p_src / "labels" / "test"
                # handle billboardXX subdir inside test/
                if (p_img / bb).exists(): p_img = p_img / bb; p_lbl = p_lbl / bb

                # Clean + defense
                apply_defense_to_dir(
                    src_img_dir = clean_root / "images" / bb,
                    src_lbl_dir = clean_root / "labels" / bb,
                    dst_img_dir = def_clean_root / "images" / bb,
                    dst_lbl_dir = def_clean_root / "labels" / bb,
                    defense_fn  = DEFENSES[def_name],
                    params      = params,
                )
                # Patched + defense
                apply_defense_to_dir(
                    src_img_dir = p_img,
                    src_lbl_dir = p_lbl,
                    dst_img_dir = def_patched_root / f"patched_dataset{i}" / "images" / "test",
                    dst_lbl_dir = def_patched_root / f"patched_dataset{i}" / "labels" / "test",
                    defense_fn  = DEFENSES[def_name],
                    params      = params,
                )

            # ── Evaluate ────────────────────────────────────────────────────
            out_json = run_eval(
                run_name       = label,
                clean_root     = def_clean_root,
                patched_parent = def_patched_root,
                scenarios_str  = FINAL_SCENARIOS,
                out_root       = OUT_ROOT,
                model_path     = model_path,
                device         = DEVICE,
            )
        except Exception as e:
            print(f"  ❌ FAILED: {e}"); continue

        m = extract_metrics_full(out_json)

        if model_none_ref is None:
            model_none_ref = m.copy()
            dc = dp = 0.0
            dd = df = {t: 0.0 for t in ["0001","03","05","07"]}
        else:
            dc = round(m["clean_mAP50"]   - model_none_ref["clean_mAP50"],   4)
            dp = round(m["patched_mAP50"] - model_none_ref["patched_mAP50"], 4)
            dd = {t: round(m[f"det_{t}"] - model_none_ref[f"det_{t}"], 3)
                  for t in ["0001","03","05","07"]}
            df = {t: round(m[f"fp_{t}"]  - model_none_ref[f"fp_{t}"],  3)
                  for t in ["0001","03","05","07"]}

        row = dict(
            model=model_key, defense=def_name, tag=tag,
            params=json.dumps(params, sort_keys=True),
            clean_mAP50=m["clean_mAP50"],   patched_mAP50=m["patched_mAP50"],
            clean_mAP5095=m["clean_mAP5095"], patched_mAP5095=m["patched_mAP5095"],
            delta_clean_mAP50=dc, delta_patched_mAP50=dp,
            elapsed_s=round(time.time()-t0, 1),
        )
        for t in ["0001","03","05","07"]:
            row[f"det@{t}"]       = m[f"det_{t}"]
            row[f"fp@{t}"]        = m[f"fp_{t}"]
            row[f"delta_det@{t}"] = dd[t] if isinstance(dd, dict) else 0.0
            row[f"delta_fp@{t}"]  = df[t] if isinstance(df, dict) else 0.0

        all_rows.append(row)
        print(f"  clean mAP50={m['clean_mAP50']:.4f} ({dc:+.4f})  "
              f"patched mAP50={m['patched_mAP50']:.4f} ({dp:+.4f})")
        print(f"  Det@0.001={m['det_0001']:.2f}  FP@0.001={m['fp_0001']:.2f}  "
              f"FP@0.3={m['fp_03']:.3f}  FP@0.5={m['fp_05']:.3f}")

final_df = pd.DataFrame(all_rows)
csv_path = Path(OUT_ROOT) / "ALL_MODELS_final_bill4to9_mAP50.csv"
final_df.to_csv(csv_path, index=False)
print(f"\n✅ Saved: {csv_path}")
display(final_df.sort_values(["model","det@0001"]).reset_index(drop=True))

## Per-Scenario Breakdown Table

In [ ]:
scen_rows = []

for model_key in ACTIVE_MODELS:
    for def_name, params in FINAL_CONFIGS:
        tag     = _tag(def_name, params)
        label   = f"{model_key}_FINAL_bill4to9_{tag}"
        cm_path = Path(OUT_ROOT) / label / "combined_metrics.json"
        if not cm_path.exists(): continue

        data = json.loads(cm_path.read_text())
        for s in data["per_scenario"]:
            if "clean" not in s or "patched" not in s: continue
            scen_rows.append({
                "model":         model_key,
                "defense":       tag,
                "scenario":      s["scenario"],
                "mAP50_C":       s["clean"]["yolo_val"].get("mAP50"),
                "mAP50_P":       s["patched"]["yolo_val"].get("mAP50"),
                "Δ_mAP50":       s.get("delta_patched_minus_clean",{}).get("yolo_val",{}).get("mAP50"),
                "mAP5095_C":     s["clean"]["yolo_val"].get("mAP50-95"),
                "mAP5095_P":     s["patched"]["yolo_val"].get("mAP50-95"),
                "Precision_C":   s["clean"]["yolo_val"].get("precision"),
                "Precision_P":   s["patched"]["yolo_val"].get("precision"),
                "Recall_C":      s["clean"]["yolo_val"].get("recall"),
                "Recall_P":      s["patched"]["yolo_val"].get("recall"),
                "Det_img_C":     s["clean"]["A"].get("detections_per_image_mean"),
                "Det_img_P":     s["patched"]["A"].get("detections_per_image_mean"),
                "Δ_Det_img":     s.get("delta_patched_minus_clean",{}).get("A",{}).get("detections_per_image_mean"),
                "FP_img_C":      s["clean"]["B"].get("fp_per_image_mean_conf_min"),
                "FP_img_P":      s["patched"]["B"].get("fp_per_image_mean_conf_min"),
                "Δ_FP_img":      s.get("delta_patched_minus_clean",{}).get("B",{}).get("fp_per_image_mean_conf_min"),
                "FP@0.3_P":      s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.3"),
                "FP@0.5_P":      s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.5"),
                "FP@0.7_P":      s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.7"),
            })

scen_df = pd.DataFrame(scen_rows)
scen_csv = Path(OUT_ROOT) / "ALL_MODELS_per_scenario_mAP50.csv"
scen_df.to_csv(scen_csv, index=False)
print(f"✅ Per-scenario CSV: {scen_csv}")

# Show per model
for model_key in ACTIVE_MODELS:
    print(f"\n── {model_key} ──")
    display(scen_df[scen_df["model"]==model_key][["defense","scenario",
        "mAP50_C","mAP50_P","Δ_mAP50","Det_img_P","Δ_Det_img","FP_img_P","Δ_FP_img"]].reset_index(drop=True))

## Save Everything to Drive

In [ ]:
import shutil

RESULTS_DIR = Path("/content/drive/MyDrive/THESIS_DEFENSE_RESULTS_v2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Master CSVs
for csv in ["ALL_MODELS_final_bill4to9_mAP50.csv",
            "ALL_MODELS_per_scenario_mAP50.csv"]:
    src = Path(OUT_ROOT) / csv
    if src.exists():
        shutil.copy2(src, RESULTS_DIR / csv)
        print(f"✅ {csv}")

# Per-model CSVs
if 'final_df' in dir():
    for model_key in ACTIVE_MODELS:
        mdf = final_df[final_df["model"]==model_key]
        p   = RESULTS_DIR / f"{model_key}_final_results_mAP50.csv"
        mdf.to_csv(p, index=False)
        print(f"✅ {p.name}")

# Raw JSONs
json_dir = RESULTS_DIR / "raw_jsons"
json_dir.mkdir(exist_ok=True)
for json_path in Path(OUT_ROOT).rglob("combined_metrics.json"):
    if "FINAL_bill4to9" in json_path.parent.name:
        dst = json_dir / f"{json_path.parent.name}.json"
        shutil.copy2(json_path, dst)
        print(f"  📄 {dst.name}")

print(f"\n📁 All results saved to: {RESULTS_DIR}")

In [ ]:
# ── Quick check: run yolov8n on billboard01-03 with no defense ──────────────
out_bb123 = run_eval(
    run_name       = "yolov8n_CHECK_bill1to3_none",
    clean_root     = CLEAN_ROOT,
    patched_parent = PATCHED_PARENT,
    scenarios_str  = "1,2,3",
    out_root       = OUT_ROOT,
    model_path     = MODEL_PATHS["yolov8n"],
    device         = DEVICE,
)
c123, p123, d123, f123 = extract_metrics(out_bb123)
print(f"bb01-03:  clean={c123:.4f}  patched={p123:.4f}")

# ── Your already-computed bb04-09 ───────────────────────────────────────────
p49 = 0.2683  # from your CSV

# ── Weighted mean across all 9 billboards ───────────────────────────────────
p_all9 = (3 * p123 + 6 * p49) / 9
c_all9 = (3 * c123 + 6 * 0.7784) / 9
print(f"\nCombined all 9:  clean={c_all9:.4f}  patched={p_all9:.4f}")
print(f"Spreadsheet:     clean=0.8064        patched=0.4918")
print(f"\nMatch? {'✅' if abs(p_all9 - 0.4918) < 0.05 else '❌ still off — datasets may differ'}")

In [ ]:
import shutil
from pathlib import Path

# 1. Delete Defended_Final_v2 (wrong labels inside patched folders)
d = Path("/content/drive/MyDrive/Patched Datasets/Defended_Final_v2")
if d.exists():
    shutil.rmtree(d)
    print("🗑️  Deleted Defended_Final_v2")
DEFENDED_ROOT = d
DEFENDED_ROOT.mkdir(parents=True, exist_ok=True)

# 2. Delete stale eval output JSONs
for folder in Path(OUT_ROOT).glob("*_FINAL_bill4to9_*"):
    if folder.is_dir():
        shutil.rmtree(folder)
        print(f"🗑️  {folder.name}")

print("✅ Clean. Ready to regenerate.")

In [ ]:
import time

final_scen_list = [int(x) for x in FINAL_SCENARIOS.split(",")]
clean_root      = Path(CLEAN_ROOT)
all_rows        = []

for model_key in ACTIVE_MODELS:
    model_path     = MODEL_PATHS[model_key]
    model_none_ref = None

    print(f"\n{'#'*70}\n  FINAL — {model_key}\n{'#'*70}")

    for def_name, params in FINAL_CONFIGS:
        tag   = _tag(def_name, params)
        label = f"{model_key}_FINAL_bill4to9_{tag}"
        print(f"\n{'='*60}\n  {label}\n{'='*60}")
        t0 = time.time()

        def_dir          = DEFENDED_ROOT / f"{model_key}_{tag}"
        def_clean_root   = def_dir / "clean"
        def_patched_root = def_dir / "patched"

        try:
            for i in final_scen_list:
                bb    = f"billboard{i:02d}"
                p_src = Path(PATCHED_PARENT) / f"patched_dataset{i}"
                p_img = p_src / "images" / "test"
                p_lbl = p_src / "labels" / "test"
                if (p_img / bb).exists(): p_img = p_img / bb; p_lbl = p_lbl / bb

                # Clean + defense
                apply_defense_to_dir(
                    src_img_dir = clean_root / "images" / bb,
                    src_lbl_dir = clean_root / "labels" / bb,
                    dst_img_dir = def_clean_root / "images" / bb,
                    dst_lbl_dir = def_clean_root / "labels" / bb,
                    defense_fn  = DEFENSES[def_name],
                    params      = params,
                )

                # Patched images + CLEAN labels (KEY FIX)
                apply_defense_to_dir(
                    src_img_dir = p_img,
                    src_lbl_dir = clean_root / "labels" / bb,  # ← clean labels
                    dst_img_dir = def_patched_root / f"patched_dataset{i}" / "images" / "test",
                    dst_lbl_dir = def_patched_root / f"patched_dataset{i}" / "labels" / "test",
                    defense_fn  = DEFENSES[def_name],
                    params      = params,
                )

            out_json = run_eval(
                run_name       = label,
                clean_root     = def_clean_root,
                patched_parent = def_patched_root,
                scenarios_str  = FINAL_SCENARIOS,
                out_root       = OUT_ROOT,
                model_path     = model_path,
                device         = DEVICE,
            )
        except Exception as e:
            print(f"  ❌ FAILED: {e}"); continue

        m = extract_metrics_full(out_json)

        if model_none_ref is None:
            model_none_ref = m.copy()
            dc = dp = 0.0
            dd = df = {t: 0.0 for t in ["0001","03","05","07"]}
        else:
            dc = round(m["clean_mAP50"]   - model_none_ref["clean_mAP50"],   4)
            dp = round(m["patched_mAP50"] - model_none_ref["patched_mAP50"], 4)
            dd = {t: round(m[f"det_{t}"] - model_none_ref[f"det_{t}"], 3)
                  for t in ["0001","03","05","07"]}
            df = {t: round(m[f"fp_{t}"]  - model_none_ref[f"fp_{t}"],  3)
                  for t in ["0001","03","05","07"]}

        row = dict(
            model=model_key, defense=def_name, tag=tag,
            params=json.dumps(params, sort_keys=True),
            clean_mAP50=m["clean_mAP50"],     patched_mAP50=m["patched_mAP50"],
            clean_mAP5095=m["clean_mAP5095"], patched_mAP5095=m["patched_mAP5095"],
            delta_clean_mAP50=dc, delta_patched_mAP50=dp,
            elapsed_s=round(time.time()-t0, 1),
        )
        for t in ["0001","03","05","07"]:
            row[f"det@{t}"]       = m[f"det_{t}"]
            row[f"fp@{t}"]        = m[f"fp_{t}"]
            row[f"delta_det@{t}"] = dd[t] if isinstance(dd, dict) else 0.0
            row[f"delta_fp@{t}"]  = df[t] if isinstance(df, dict) else 0.0

        all_rows.append(row)
        print(f"  clean={m['clean_mAP50']:.4f}({dc:+.4f})  "
              f"patched={m['patched_mAP50']:.4f}({dp:+.4f})")
        print(f"  Det@0.001={m['det_0001']:.2f}  FP@0.001={m['fp_0001']:.2f}  "
              f"FP@0.3={m['fp_03']:.3f}  FP@0.5={m['fp_05']:.3f}")

final_df = pd.DataFrame(all_rows)
csv_path = Path(OUT_ROOT) / "ALL_MODELS_final_bill4to9_mAP50.csv"
final_df.to_csv(csv_path, index=False)
print(f"\n✅ Saved: {csv_path}")
display(final_df.sort_values(["model","det@0001"]).reset_index(drop=True))  Det@0.001=63.25  FP@0.001=59.57  FP@0.3=6.277  FP@0.5=1.593


In [ ]:
import shutil, json
import pandas as pd
from pathlib import Path

OUT_ROOT    = Path(OUT_ROOT)
RESULTS_DIR = Path("/content/drive/MyDrive/THESIS_DEFENSE_RESULTS_v2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Per-scenario CSV ────────────────────────────────────────────────────────
scen_rows = []
for model_key in ACTIVE_MODELS:
    for def_name, params in FINAL_CONFIGS:
        tag     = _tag(def_name, params)
        label   = f"{model_key}_FINAL_bill4to9_{tag}"
        cm_path = OUT_ROOT / label / "combined_metrics.json"
        if not cm_path.exists(): continue

        data = json.loads(cm_path.read_text())
        for s in data["per_scenario"]:
            if "clean" not in s or "patched" not in s: continue
            scen_rows.append({
                "model":       model_key,
                "defense":     tag,
                "scenario":    s["scenario"],
                "mAP50_C":     s["clean"]["yolo_val"].get("mAP50"),
                "mAP50_P":     s["patched"]["yolo_val"].get("mAP50"),
                "Δ_mAP50":     s.get("delta_patched_minus_clean",{}).get("yolo_val",{}).get("mAP50"),
                "mAP5095_C":   s["clean"]["yolo_val"].get("mAP50-95"),
                "mAP5095_P":   s["patched"]["yolo_val"].get("mAP50-95"),
                "Precision_C": s["clean"]["yolo_val"].get("precision"),
                "Precision_P": s["patched"]["yolo_val"].get("precision"),
                "Recall_C":    s["clean"]["yolo_val"].get("recall"),
                "Recall_P":    s["patched"]["yolo_val"].get("recall"),
                "Det_img_C":   s["clean"]["A"].get("detections_per_image_mean"),
                "Det_img_P":   s["patched"]["A"].get("detections_per_image_mean"),
                "Δ_Det_img":   s.get("delta_patched_minus_clean",{}).get("A",{}).get("detections_per_image_mean"),
                "FP_img_C":    s["clean"]["B"].get("fp_per_image_mean_conf_min"),
                "FP_img_P":    s["patched"]["B"].get("fp_per_image_mean_conf_min"),
                "Δ_FP_img":    s.get("delta_patched_minus_clean",{}).get("B",{}).get("fp_per_image_mean_conf_min"),
                "FP@0.3_P":    s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.3"),
                "FP@0.5_P":    s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.5"),
                "FP@0.7_P":    s["patched"]["B"].get("fp_per_image_conf_ge",{}).get("0.7"),
            })

scen_df  = pd.DataFrame(scen_rows)
scen_csv = RESULTS_DIR / "ALL_MODELS_per_scenario_mAP50.csv"
scen_df.to_csv(scen_csv, index=False)
print(f"✅ {scen_csv.name}")

# ── Summary CSV ─────────────────────────────────────────────────────────────
summary_csv = RESULTS_DIR / "ALL_MODELS_final_bill4to9_mAP50.csv"
if 'final_df' in dir() and len(final_df) > 0:
    final_df.to_csv(summary_csv, index=False)
    print(f"✅ {summary_csv.name}")

    # Per-model CSVs
    for model_key in ACTIVE_MODELS:
        mdf = final_df[final_df["model"] == model_key]
        p   = RESULTS_DIR / f"{model_key}_final_results_mAP50.csv"
        mdf.to_csv(p, index=False)
        print(f"✅ {p.name}")

# ── Raw JSONs ───────────────────────────────────────────────────────────────
json_dir = RESULTS_DIR / "raw_jsons"
json_dir.mkdir(exist_ok=True)
for json_path in OUT_ROOT.rglob("combined_metrics.json"):
    if "FINAL_bill4to9" in json_path.parent.name:
        dst = json_dir / f"{json_path.parent.name}.json"
        shutil.copy2(json_path, dst)
        print(f"  📄 {dst.name}")

print(f"\n📁 All saved to: {RESULTS_DIR}")
display(scen_df.groupby(["model","defense"])[["mAP50_C","mAP50_P","Det_img_P","FP_img_P"]].mean().round(3))

In [ ]:
# CELL A — Precomputed LGS held-out dataset paths

from pathlib import Path

# Adjust only if your Drive folder names differ slightly
LGS_DATASET_ROOT = Path(
    "/content/drive/MyDrive/Patched Datasets/Defended_LGS_SELECTED/LGS_HELDOUT_selected_bill4to9"
)

LGS_CLEAN_ROOT   = LGS_DATASET_ROOT / "clean"
LGS_PATCHED_ROOT = LGS_DATASET_ROOT / "patched"
LGS_TAG          = "lgs_selected_bill4to9"

print("LGS_DATASET_ROOT:", LGS_DATASET_ROOT)
print("LGS_CLEAN_ROOT  :", LGS_CLEAN_ROOT,   "| exists:", LGS_CLEAN_ROOT.exists())
print("LGS_PATCHED_ROOT:", LGS_PATCHED_ROOT, "| exists:", LGS_PATCHED_ROOT.exists())

assert LGS_CLEAN_ROOT.exists(),   "LGS clean root not found. Fix the path."
assert LGS_PATCHED_ROOT.exists(), "LGS patched root not found. Fix the path."

In [ ]:
# CELL B — Sanity check that the evaluator will find the LGS folders

def first_existing(*paths):
    for p in paths:
        if p.exists():
            return p
    return None

for i in [4, 5, 6, 7, 8, 9]:
    bb = f"billboard{i:02d}"

    clean_img = first_existing(
        LGS_CLEAN_ROOT / "images" / bb,
        LGS_CLEAN_ROOT / "images" / "test" / bb,
        LGS_CLEAN_ROOT / "images" / "test",
        LGS_CLEAN_ROOT / "images",
    )
    clean_lbl = first_existing(
        LGS_CLEAN_ROOT / "labels" / bb,
        LGS_CLEAN_ROOT / "labels" / "test" / bb,
        LGS_CLEAN_ROOT / "labels" / "test",
        LGS_CLEAN_ROOT / "labels",
    )

    patched_img = first_existing(
        LGS_PATCHED_ROOT / "images" / bb,
        LGS_PATCHED_ROOT / "images" / "test" / bb,
        LGS_PATCHED_ROOT / "images" / "test",
        LGS_PATCHED_ROOT / "images",
    )
    patched_lbl = first_existing(
        LGS_PATCHED_ROOT / "labels" / bb,
        LGS_PATCHED_ROOT / "labels" / "test" / bb,
        LGS_PATCHED_ROOT / "labels" / "test",
        LGS_PATCHED_ROOT / "labels",
    )

    print(f"\n{bb}")
    print("  clean   images:", clean_img)
    print("  clean   labels:", clean_lbl)
    print("  patched images:", patched_img)
    print("  patched labels:", patched_lbl)

    assert clean_img is not None,   f"Missing clean images for {bb}"
    assert clean_lbl is not None,   f"Missing clean labels for {bb}"
    assert patched_img is not None, f"Missing patched images for {bb}"
    assert patched_lbl is not None, f"Missing patched labels for {bb}"

print("\n✅ Folder structure looks usable.")
print("⚠️ Make sure patched labels are the CLEAN ground-truth labels for fair evaluation.")

In [ ]:
from pathlib import Path
import shutil

def ensure_dir(p):
    p.mkdir(parents=True, exist_ok=True)

for i in [4,5,6,7,8,9]:
    bb = f"billboard{i:02d}"

    clean_lbl_dir   = resolve_label_dir_like_eval(LGS_CLEAN_ROOT, bb)
    patched_lbl_dir = resolve_label_dir_like_eval(LGS_PATCHED_ROOT, bb)

    assert clean_lbl_dir is not None, f"Missing clean labels for {bb}"

    # Prefer billboard-specific target if it exists, otherwise create one under labels/test/billboardXX
    if patched_lbl_dir is None:
        patched_lbl_dir = LGS_PATCHED_ROOT / "labels" / "test" / bb

    if patched_lbl_dir.exists():
        shutil.rmtree(patched_lbl_dir)

    ensure_dir(patched_lbl_dir)

    for txt in clean_lbl_dir.rglob("*.txt"):
        dst = patched_lbl_dir / txt.name
        shutil.copy2(txt, dst)

    print(f"✅ Replaced patched labels with clean GT for {bb}: {patched_lbl_dir}")

In [ ]:
from pathlib import Path
import hashlib

LGS_DATASET_ROOT = Path("/content/drive/MyDrive/Patched Datasets/Defended_LGS_SELECTED/LGS_HELDOUT_selected_bill4to9")
LGS_CLEAN_ROOT   = LGS_DATASET_ROOT / "clean"
LGS_PATCHED_ROOT = LGS_DATASET_ROOT / "patched"

def resolve_label_dir_like_eval(root, billboard_name):
    candidates = [
        root / "labels" / billboard_name,
        root / "labels" / "test" / billboard_name,
        root / "labels" / "test",
        root / "labels",
    ]
    for d in candidates:
        if d.exists():
            txts = list(d.rglob("*.txt"))
            if txts:
                return d
    return None

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

all_ok = True

for i in [4,5,6,7,8,9]:
    bb = f"billboard{i:02d}"
    clean_lbl_dir   = resolve_label_dir_like_eval(LGS_CLEAN_ROOT, bb)
    patched_lbl_dir = resolve_label_dir_like_eval(LGS_PATCHED_ROOT, bb)

    print(f"\n=== {bb} ===")
    print("clean_lbl_dir  :", clean_lbl_dir)
    print("patched_lbl_dir:", patched_lbl_dir)

    assert clean_lbl_dir is not None, f"Missing clean labels for {bb}"
    assert patched_lbl_dir is not None, f"Missing patched labels for {bb}"

    clean_files   = sorted([p for p in clean_lbl_dir.rglob("*.txt")])
    patched_files = sorted([p for p in patched_lbl_dir.rglob("*.txt")])

    clean_names   = [p.name for p in clean_files]
    patched_names = [p.name for p in patched_files]

    same_names = clean_names == patched_names
    print("same filename list:", same_names)
    print("clean count:", len(clean_files), "| patched count:", len(patched_files))

    if not same_names:
        all_ok = False
        print("❌ Label filenames differ")
        continue

    diff_count = 0
    for c, p in zip(clean_files, patched_files):
        if md5(c) != md5(p):
            diff_count += 1

    print("different file contents:", diff_count)

    if diff_count > 0:
        all_ok = False
        print("❌ Patched labels are NOT identical to clean GT")
    else:
        print("✅ Patched labels match clean GT")

print("\nFINAL:", "✅ patched labels are clean GT" if all_ok else "❌ patched labels are NOT clean GT")

In [ ]:
# CELL C — Evaluate undefended baseline vs precomputed LGS for all models

import time
import json
import pandas as pd
from pathlib import Path

baseline_jsons = {}
lgs_jsons = {}
rows = []

for model_key in ACTIVE_MODELS:
    model_path = MODEL_PATHS[model_key]

    print(f"\n{'#'*80}")
    print(f"MODEL: {model_key}")
    print(f"{'#'*80}")

    # 1) Undefended baseline on the original held-out data
    t0 = time.time()
    baseline_json = run_eval(
        run_name       = f"{model_key}_FINAL_bill4to9_none_baseline",
        clean_root     = CLEAN_ROOT,
        patched_parent = PATCHED_PARENT,
        scenarios_str  = FINAL_SCENARIOS,
        out_root       = OUT_ROOT,
        model_path     = model_path,
        device         = DEVICE,
    )
    t1 = time.time()

    # 2) Precomputed LGS defended held-out dataset
    t2 = time.time()
    lgs_json = run_eval(
        run_name      = f"{model_key}_FINAL_bill4to9_{LGS_TAG}",
        clean_root    = str(LGS_CLEAN_ROOT),
        patched_root  = str(LGS_PATCHED_ROOT),
        scenarios_str = FINAL_SCENARIOS,
        out_root      = OUT_ROOT,
        model_path    = model_path,
        device        = DEVICE,
    )
    t3 = time.time()

    baseline_jsons[model_key] = baseline_json
    lgs_jsons[model_key] = lgs_json

    base = extract_metrics_full(baseline_json)
    lgs  = extract_metrics_full(lgs_json)

    row = {
        "model": model_key,

        "baseline_clean_mAP50": base["clean_mAP50"],
        "baseline_patched_mAP50": base["patched_mAP50"],
        "lgs_clean_mAP50": lgs["clean_mAP50"],
        "lgs_patched_mAP50": lgs["patched_mAP50"],
        "delta_clean_mAP50": round(lgs["clean_mAP50"] - base["clean_mAP50"], 4),
        "delta_patched_mAP50": round(lgs["patched_mAP50"] - base["patched_mAP50"], 4),

        "baseline_clean_mAP5095": base["clean_mAP5095"],
        "baseline_patched_mAP5095": base["patched_mAP5095"],
        "lgs_clean_mAP5095": lgs["clean_mAP5095"],
        "lgs_patched_mAP5095": lgs["patched_mAP5095"],
        "delta_clean_mAP5095": round(lgs["clean_mAP5095"] - base["clean_mAP5095"], 4),
        "delta_patched_mAP5095": round(lgs["patched_mAP5095"] - base["patched_mAP5095"], 4),

        "baseline_elapsed_s": round(t1 - t0, 1),
        "lgs_elapsed_s": round(t3 - t2, 1),
    }

    for t in ["0001", "03", "05", "07"]:
        row[f"baseline_det@{t}"] = base[f"det_{t}"]
        row[f"baseline_fp@{t}"]  = base[f"fp_{t}"]
        row[f"lgs_det@{t}"]      = lgs[f"det_{t}"]
        row[f"lgs_fp@{t}"]       = lgs[f"fp_{t}"]
        row[f"delta_det@{t}"]    = round(lgs[f"det_{t}"] - base[f"det_{t}"], 3)
        row[f"delta_fp@{t}"]     = round(lgs[f"fp_{t}"] - base[f"fp_{t}"], 3)

    rows.append(row)

    print(f"baseline patched mAP50 : {base['patched_mAP50']:.4f}")
    print(f"LGS      patched mAP50 : {lgs['patched_mAP50']:.4f} ({row['delta_patched_mAP50']:+.4f})")
    print(f"baseline FP@0.001      : {base['fp_0001']:.3f}")
    print(f"LGS      FP@0.001      : {lgs['fp_0001']:.3f} ({row['delta_fp@0001']:+.3f})")

lgs_all_models_df = pd.DataFrame(rows).sort_values(
    ["delta_patched_mAP50", "delta_fp@0001"],
    ascending=[False, True]
).reset_index(drop=True)

display(lgs_all_models_df)

In [ ]:
# CELL D — Per-scenario comparison for each model

import json
import pandas as pd
from pathlib import Path

raw_rows = []

for model_key in ACTIVE_MODELS:
    for version, json_path in [
        ("baseline", baseline_jsons[model_key]),
        ("lgs",      lgs_jsons[model_key]),
    ]:
        data = json.loads(Path(json_path).read_text())

        for s in data.get("per_scenario", []):
            if "clean" not in s or "patched" not in s:
                continue

            raw_rows.append({
                "model": model_key,
                "version": version,
                "scenario": s["scenario"],

                "clean_mAP50":   s["clean"]["yolo_val"].get("mAP50"),
                "patched_mAP50": s["patched"]["yolo_val"].get("mAP50"),

                "clean_mAP5095":   s["clean"]["yolo_val"].get("mAP50-95"),
                "patched_mAP5095": s["patched"]["yolo_val"].get("mAP50-95"),

                "det@0001": s["patched"]["A"].get("detections_per_image_mean"),
                "fp@0001":  s["patched"]["B"].get("fp_per_image_mean_conf_min"),
                "fp@03":    s["patched"]["B"].get("fp_per_image_conf_ge", {}).get("0.3"),
                "fp@05":    s["patched"]["B"].get("fp_per_image_conf_ge", {}).get("0.5"),
                "fp@07":    s["patched"]["B"].get("fp_per_image_conf_ge", {}).get("0.7"),
            })

raw_df = pd.DataFrame(raw_rows)

base_df = raw_df[raw_df["version"] == "baseline"].drop(columns=["version"]).rename(columns={
    "clean_mAP50": "baseline_clean_mAP50",
    "patched_mAP50": "baseline_patched_mAP50",
    "clean_mAP5095": "baseline_clean_mAP5095",
    "patched_mAP5095": "baseline_patched_mAP5095",
    "det@0001": "baseline_det@0001",
    "fp@0001": "baseline_fp@0001",
    "fp@03": "baseline_fp@03",
    "fp@05": "baseline_fp@05",
    "fp@07": "baseline_fp@07",
})

lgs_df = raw_df[raw_df["version"] == "lgs"].drop(columns=["version"]).rename(columns={
    "clean_mAP50": "lgs_clean_mAP50",
    "patched_mAP50": "lgs_patched_mAP50",
    "clean_mAP5095": "lgs_clean_mAP5095",
    "patched_mAP5095": "lgs_patched_mAP5095",
    "det@0001": "lgs_det@0001",
    "fp@0001": "lgs_fp@0001",
    "fp@03": "lgs_fp@03",
    "fp@05": "lgs_fp@05",
    "fp@07": "lgs_fp@07",
})

lgs_per_scenario_df = base_df.merge(lgs_df, on=["model", "scenario"], how="inner")

lgs_per_scenario_df["delta_clean_mAP50"]   = (lgs_per_scenario_df["lgs_clean_mAP50"]   - lgs_per_scenario_df["baseline_clean_mAP50"]).round(4)
lgs_per_scenario_df["delta_patched_mAP50"] = (lgs_per_scenario_df["lgs_patched_mAP50"] - lgs_per_scenario_df["baseline_patched_mAP50"]).round(4)
lgs_per_scenario_df["delta_det@0001"]      = (lgs_per_scenario_df["lgs_det@0001"]      - lgs_per_scenario_df["baseline_det@0001"]).round(3)
lgs_per_scenario_df["delta_fp@0001"]       = (lgs_per_scenario_df["lgs_fp@0001"]       - lgs_per_scenario_df["baseline_fp@0001"]).round(3)
lgs_per_scenario_df["delta_fp@03"]         = (lgs_per_scenario_df["lgs_fp@03"]         - lgs_per_scenario_df["baseline_fp@03"]).round(3)
lgs_per_scenario_df["delta_fp@05"]         = (lgs_per_scenario_df["lgs_fp@05"]         - lgs_per_scenario_df["baseline_fp@05"]).round(3)
lgs_per_scenario_df["delta_fp@07"]         = (lgs_per_scenario_df["lgs_fp@07"]         - lgs_per_scenario_df["baseline_fp@07"]).round(3)

display(lgs_per_scenario_df.sort_values(["model", "scenario"]).reset_index(drop=True))

In [ ]:
# CELL E — Save LGS comparison results

from pathlib import Path

LGS_RESULTS_DIR = Path("/content/drive/MyDrive/THESIS_DEFENSE_RESULTS_LGS")
LGS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

summary_path = LGS_RESULTS_DIR / "LGS_all_models_summary.csv"
scenario_path = LGS_RESULTS_DIR / "LGS_per_scenario_comparison.csv"

lgs_all_models_df.to_csv(summary_path, index=False)
lgs_per_scenario_df.to_csv(scenario_path, index=False)

print("✅ Saved:", summary_path)
print("✅ Saved:", scenario_path)